### Load Data

In [50]:
import pandas as pd
from sklearn.model_selection import train_test_split

# load CSV
df = pd.read_csv("data/label/trainLabels.csv")

# adjust depending on your CSV format
df["image_path"] = df["image"].apply(lambda x: f"data/images/{x}.jpeg")
df["level"] = df["level"]

# split
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["level"],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

In [51]:
print("Matched images:", len(df))
print(df["level"].value_counts().sort_index())
df.head()

Matched images: 35126
level
0    25810
1     2443
2     5292
3      873
4      708
Name: count, dtype: int64


,image,level,image_path
0,10_left,0,data/images/10_left.jpeg
1,10_right,0,data/images/10_right.jpeg
2,13_left,0,data/images/13_left.jpeg
3,13_right,0,data/images/13_right.jpeg
4,15_left,1,data/images/15_left.jpeg


### Dataset Class

In [52]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn
from torchvision import models

from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score


class DRNpyDataset(Dataset):
    def __init__(self, dataframe, npy_dir):
        self.dataframe = dataframe.reset_index(drop=True)
        self.npy_dir = Path(npy_dir)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image_path = Path(row["image_path"])
        label = int(row["level"])

        # original image: xxx.jpeg -> normalized file: xxx.npy
        npy_path = self.npy_dir / f"{image_path.stem}.npy"

        if not npy_path.exists():
            raise FileNotFoundError(f"Missing npy file: {npy_path}")

        img = np.load(npy_path).astype(np.float32)   # shape: (H, W, C)

        # convert HWC -> CHW for PyTorch
        img = np.transpose(img, (2, 0, 1))

        # to torch tensor
        img = torch.from_numpy(img)

        return img, label

In [53]:
normalized_dir = "data/processed/normalized"

train_dataset_npy = DRNpyDataset(train_df, npy_dir=normalized_dir)
val_dataset_npy = DRNpyDataset(val_df, npy_dir=normalized_dir)

### Setting Device as GPU/MPS

In [54]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Using device:", DEVICE)

Using device: mps


### Training & Validate

In [55]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device, dtype=torch.float32)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average="weighted")
    epoch_kappa = cohen_kappa_score(all_labels, all_preds, weights="quadratic")

    return epoch_loss, epoch_acc, epoch_f1, epoch_kappa


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device, dtype=torch.float32)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average="weighted")
    epoch_kappa = cohen_kappa_score(all_labels, all_preds, weights="quadratic")

    return epoch_loss, epoch_acc, epoch_f1, epoch_kappa, all_labels, all_preds

### Kaiming Initialization

In [56]:
def init_weights_kaiming(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

### Model - ResNet18
1. `resnet18_with_npy` - with Kaiming intialization.
2. `resnet18_with_npy_v2` - with pre-trained ResNet18 default weights.

In [57]:
import copy


def resnet18_with_npy(
    lr,
    batch_size,
    num_epochs,
    train_dataset,
    val_dataset,
    device,
    num_classes=5,
    class_weights=None
):
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    model = models.resnet18(weights=None)
    model.apply(init_weights_kaiming)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=2, factor=0.5
    )

    history = []
    best_kappa = -1
    best_model_state = None
    best_epoch = -1

    for epoch in range(num_epochs):
        train_loss, train_acc, train_f1, train_kappa = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )

        val_loss, val_acc, val_f1, val_kappa, val_labels, val_preds = validate_one_epoch(
            model, val_loader, criterion, device
        )

        scheduler.step(val_kappa)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_f1": train_f1,
            "train_kappa": train_kappa,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1,
            "val_kappa": val_kappa,
        })

        print(f"[lr={lr}, bs={batch_size}] Epoch {epoch+1}/{num_epochs}")
        print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f} | Kappa: {train_kappa:.4f}")
        print(f"  Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f} | Kappa: {val_kappa:.4f}")

        if val_kappa > best_kappa:
            best_kappa = val_kappa
            best_epoch = epoch + 1
            best_model_state = copy.deepcopy(model.state_dict())

    return {
        "lr": lr,
        "batch_size": batch_size,
        "history": history,
        "best_kappa": best_kappa,
        "best_epoch": best_epoch,
        "best_model_state": best_model_state,
    }

def resnet18_with_npy_v2(
        lr,
        batch_size,
        num_epochs,
        train_dataset,
        val_dataset,
        device,
        num_classes=5,
        class_weights=None
    ):
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0
        )
        weights = models.ResNet18_Weights.DEFAULT

        model = models.resnet18(weights=weights)
        # model.apply(init_weights_kaiming)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        model = model.to(device)

        criterion = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", patience=2, factor=0.5
        )

        history = []
        best_kappa = -1
        best_model_state = None
        best_epoch = -1

        for epoch in range(num_epochs):
            train_loss, train_acc, train_f1, train_kappa = train_one_epoch(
                model, train_loader, criterion, optimizer, device
            )

            val_loss, val_acc, val_f1, val_kappa, val_labels, val_preds = validate_one_epoch(
                model, val_loader, criterion, device
            )

            scheduler.step(val_kappa)

            history.append({
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "train_f1": train_f1,
                "train_kappa": train_kappa,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "val_f1": val_f1,
                "val_kappa": val_kappa,
            })

            print(f"[lr={lr}, bs={batch_size}] Epoch {epoch+1}/{num_epochs}")
            print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f} | Kappa: {train_kappa:.4f}")
            print(f"  Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f} | Kappa: {val_kappa:.4f}")

            if val_kappa > best_kappa:
                best_kappa = val_kappa
                best_epoch = epoch + 1
                best_model_state = copy.deepcopy(model.state_dict())

        return {
            "lr": lr,
            "batch_size": batch_size,
            "history": history,
            "best_kappa": best_kappa,
            "best_epoch": best_epoch,
            "best_model_state": best_model_state,
        }

### Improve Imbalance
In EyePACS, the class distribution looks like:
- 0 (No DR)       → very large
- 1 (Mild)        → small
- 2 (Moderate)    → medium
- 3 (Severe)      → very small
- 4 (Proliferative) → very small

So without weighting:
- Model just predicts 0 (No DR) all the time and gets high accuracy but is clinically useless.

In [41]:
NUM_CLASSES = 5

all_class_counts = np.array([
    len(train_df[train_df["level"] == c]) for c in range(NUM_CLASSES)
])

all_class_counts = np.where(all_class_counts == 0, 1, all_class_counts)
class_weights = 1.0 / all_class_counts
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

### Case 1 (Learning Rate: 0.0003, Batch Size: 32)
- Weight Initialization with Kaiming Intitialization

In [ ]:
result_npy = resnet18_with_npy(
    lr=3e-4,
    batch_size=32,
    num_epochs=20,
    train_dataset=train_dataset_npy,
    val_dataset=val_dataset_npy,
    device=DEVICE,
    num_classes=NUM_CLASSES,
    class_weights=class_weights_tensor
)

[lr=0.0003, bs=32] Epoch 1/20
  Train Loss: 1.6123 | Acc: 0.3014 | F1: 0.3779 | Kappa: 0.0638
  Val   Loss: 1.5438 | Acc: 0.2328 | F1: 0.2543 | Kappa: 0.0812
[lr=0.0003, bs=32] Epoch 2/20
  Train Loss: 1.5596 | Acc: 0.3488 | F1: 0.4272 | Kappa: 0.1021
  Val   Loss: 1.5251 | Acc: 0.3479 | F1: 0.4200 | Kappa: 0.1137
[lr=0.0003, bs=32] Epoch 3/20
  Train Loss: 1.5236 | Acc: 0.3466 | F1: 0.4228 | Kappa: 0.1163
  Val   Loss: 1.5607 | Acc: 0.6018 | F1: 0.6060 | Kappa: 0.1896
[lr=0.0003, bs=32] Epoch 4/20
  Train Loss: 1.4850 | Acc: 0.3392 | F1: 0.4141 | Kappa: 0.1326
  Val   Loss: 1.4825 | Acc: 0.4328 | F1: 0.4892 | Kappa: 0.1533
[lr=0.0003, bs=32] Epoch 5/20
  Train Loss: 1.4428 | Acc: 0.3614 | F1: 0.4348 | Kappa: 0.1676
  Val   Loss: 1.3998 | Acc: 0.3147 | F1: 0.3882 | Kappa: 0.1466
[lr=0.0003, bs=32] Epoch 6/20
  Train Loss: 1.3930 | Acc: 0.3846 | F1: 0.4558 | Kappa: 0.1960
  Val   Loss: 1.4267 | Acc: 0.3641 | F1: 0.4374 | Kappa: 0.1726
[lr=0.0003, bs=32] Epoch 7/20
  Train Loss: 1.2944 |

### Case 2 (Learning Rate: 0.0003, Batch Size: 32)
- Weight Initialization with pretrained ResNet18 default weights.

In [59]:
result_npy = resnet18_with_npy_v2(
    lr=3e-4,
    batch_size=32,
    num_epochs=20,
    train_dataset=train_dataset_npy,
    val_dataset=val_dataset_npy,
    device=DEVICE,
    num_classes=NUM_CLASSES,
    class_weights=class_weights_tensor
)

[lr=0.0003, bs=32] Epoch 1/20
  Train Loss: 1.3585 | Acc: 0.3947 | F1: 0.4651 | Kappa: 0.2731
  Val   Loss: 1.3636 | Acc: 0.4081 | F1: 0.4794 | Kappa: 0.3988
[lr=0.0003, bs=32] Epoch 2/20
  Train Loss: 1.2094 | Acc: 0.4578 | F1: 0.5221 | Kappa: 0.3689
  Val   Loss: 1.2121 | Acc: 0.5340 | F1: 0.5802 | Kappa: 0.3630
[lr=0.0003, bs=32] Epoch 3/20
  Train Loss: 1.1219 | Acc: 0.4573 | F1: 0.5204 | Kappa: 0.3985
  Val   Loss: 1.2369 | Acc: 0.3841 | F1: 0.4525 | Kappa: 0.3232
[lr=0.0003, bs=32] Epoch 4/20
  Train Loss: 1.0354 | Acc: 0.4689 | F1: 0.5303 | Kappa: 0.4246
  Val   Loss: 1.3569 | Acc: 0.3296 | F1: 0.3984 | Kappa: 0.3774
[lr=0.0003, bs=32] Epoch 5/20
  Train Loss: 0.8030 | Acc: 0.5386 | F1: 0.5920 | Kappa: 0.5099
  Val   Loss: 1.2243 | Acc: 0.4862 | F1: 0.5477 | Kappa: 0.4451
[lr=0.0003, bs=32] Epoch 6/20
  Train Loss: 0.6372 | Acc: 0.5811 | F1: 0.6273 | Kappa: 0.5468
  Val   Loss: 1.3432 | Acc: 0.4451 | F1: 0.5101 | Kappa: 0.4319
[lr=0.0003, bs=32] Epoch 7/20
  Train Loss: 0.4845 |